## Import Libraries

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import f

##

# Load Dataset

In [4]:
df = pd.read_csv("health_dataset_200_records.csv")

df


,record_id,age_group,age,weight,gender,region,smoking_status,exercise_frequency,bmi,blood_pressure,diabetes,hypertension,cholesterol_level,glucose_level,visit_date
0,7b9e8694-51d3-42dd-9e52-f167f69fd0e7,46-60,49,77,Female,West,Smoker,Weekly,35.0,108.5,False,True,238.0,199.2,2025-01-26
1,099e486a-395f-4965-8309-f43afed184f7,36-45,36,57,Other,West,Smoker,Rarely,36.4,173.5,True,False,299.7,195.8,2026-02-01
2,f4aea4d3-7320-4a83-b824-f0baf02fe541,26-35,27,95,Male,North,Smoker,Daily,19.0,121.7,True,True,221.2,114.5,2024-04-29
3,ce234b08-9bf9-4f07-955f-568229890e98,46-60,46,65,Other,South,Non-Smoker,Daily,27.2,144.9,False,True,219.7,203.3,2025-11-29
4,67e5c7f4-c150-43dc-856d-bfd73cfe9024,26-35,26,106,Male,South,Former Smoker,Rarely,18.6,154.0,True,False,231.2,135.9,2024-10-22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,223ae7b8-85d1-4208-bd4b-46d012dddd5a,60+,70,86,Male,East,Smoker,Never,30.4,125.9,True,False,272.0,216.6,2024-04-21
196,a37c8874-24c3-469b-9bdd-4022aef3b700,60+,79,107,Other,South,Former Smoker,Rarely,35.2,179.1,False,False,176.8,160.4,2024-05-30
197,bc37acc5-341a-4545-996b-2dbab6987dfa,26-35,29,98,Male,West,Former Smoker,Rarely,29.6,178.1,True,False,199.8,125.4,2026-01-07
198,a375f519-13f0-4861-85f7-1fdf9cec0b1d,60+,65,67,Other,North,Former Smoker,Weekly,29.5,107.6,False,True,263.4,176.0,2025-01-07


# 1. HYPOTHESIS

In [3]:
print("\nHYPOTHESIS 1")
print("H0: Smoking status has no effict on diabetes prevalence")
print("H1: Smoking status affects diabetes prevalance")

print("\nHYPOTHESIS 2")
print("H0: No significant difference in BMI between genders")
print("H1: Significant difference in BMI between genders")


HYPOTHESIS 1
H0: Smoking status has no effict on diabetes prevalence
H1: Smoking status affects diabetes prevalance

HYPOTHESIS 2
H0: No significant difference in BMI between genders
H1: Significant difference in BMI between genders


# 2. CONFIDENCE INTERVALS

In [5]:

for col in ['age', 'weight']:

    mean = df[col].mean()
    sem = stats.sem(df[col])

    ci = stats.t.interval(
        confidence=0.95,
        df=len(df[col])-1,
        loc=mean,
        scale = sem
    )

    print(f"\n{col.upper()}")
    print("Mean:", round(mean,2))
    print("95% Confidence Intervals:", ci)


AGE
Mean: 49.04
95% Confidence Intervals: (np.float64(46.46748145499341), np.float64(51.60251854500658))

WEIGHT
Mean: 78.0
95% Confidence Intervals: (np.float64(75.22625659028515), np.float64(80.76374340971486))


# 3 & 4. T-TEST (BMI by Gender)

In [6]:


male_bmi = df[df['gender']=='Male']['bmi']
female_bmi = df[df['gender']=='Female']['bmi']

t_stat, p_value = stats.ttest_ind(
    male_bmi,
    female_bmi,
    equal_var= False
)

critical_value = stats.t.ppf(
    1-0.025,
    len(df)-2
)

print("T Statistic = ",t_stat)
print("P_value=",p_value)
print("Critical Value=",critical_value)

if p_value < 0.05:
    print("Reject H0")
else:
    print("Accept H0")


T Statistic =  -1.1518561654828081
P_value= 0.25164826399277224
Critical Value= 1.9720174778363146
Accept H0


# 5. CHI-SQUARE TEST

In [7]:

contingency = pd.crosstab(
    df['smoking_status'],
    df['diabetes']
)

chi2, p, dof, expected = stats.chi2_contingency(contingency)

critical = stats.chi2.ppf(
    0.95,
    dof
)

print("Chi-Square Statistic = ",chi2)
print("P value=",p)
print("critical_value=",critical)

if p_value < 0.05:
    print("Reject H0")
else:
    print("Accept H0")

Chi-Square Statistic =  0.028911394281340725
P value= 0.9856482848024194
critical_value= 5.991464547107979
Accept H0


# 6. ANOVA TEST

In [8]:

groups = df.groupby('age_group')['bmi']

k = len(groups)

n = len(df)

grand_mean = df['bmi'].mean()

print("Grand Mean =", grand_mean)

SSB = 0

for name, group in groups:
    n=len(group)
    mean = group.mean()

    print(f"\n{name}")
    print("n =",n)
    print("Mean =", mean)

    SSB += n * ((mean -grand_mean) ** 2)

print("\nSSB =", SSB)

SSW = 0

for name, group in groups:
    mean = group.mean()

    for value in group:
        SSW += (value - mean) ** 2

print("SSW =", SSW)


df_between = k - 1
df_within = n - k

print("\ndf_between =", df_between)
print("df_within =",df_within )


MSB = SSB / df_between
MSW = SSW / df_within

print("\nMSB =", MSB)
print("MWS =", MSW)


F_stat = MSB / MSW

print("\nF statistic =", F_stat)


alpha = 0.05

F_critical = f.ppf(
    1 - alpha,
    df_between,
    df_within
)

print("F Critical =",F_critical)


p_value = 1 - f.cdf(
    F_stat,
    df_between,
    df_within
)

print("P Value =", p_value)


print("\nDecison")

if p_value < 0.05:
    print("Reject H0")
    print("BMI differs significantly among age groups.")
else:
    print("Accept H0")
    print("No singificantly difference among age groups.")

Grand Mean = 27.593999999999998

18-25
n = 22
Mean = 27.518181818181816

26-35
n = 38
Mean = 27.89736842105263

36-45
n = 38
Mean = 25.03421052631579

46-60
n = 39
Mean = 30.007692307692306

60+
n = 63
Mean = 27.48730158730159

SSB = 480.54727599184406
SSW = 6200.245524008152

df_between = 4
df_within = 58

MSB = 120.13681899796102
MWS = 106.90078489669227

F statistic = 1.1238160609771004
F Critical = 2.530694205468098
P Value = 0.35418979784033167

Decison
Accept H0
No singificantly difference among age groups.


# 7. COVARIANCE & CORRELATION

In [8]:
convariance = np.cov(
    df['age'],
    df['bmi']
)[0][1]

correlation = df['age'].corr(df['bmi'])

print("Convariance =",convariance)
print("Correlaton =", correlation)


Convariance = 5.781618090452267
Correlaton = 0.05419143991028517


# 8. FINAL SUMMARY

In [9]:
print("✅ Confidence Interval Calculated")
print("✅ t-Test Performed")
print("✅ Chi-Square Test Performed")
print("✅ ANOVA Test Performed")
print("✅ Covariance Calculated")
print("✅ Correlation Calulated")

✅ Confidence Interval Calculated
✅ t-Test Performed
✅ Chi-Square Test Performed
✅ ANOVA Test Performed
✅ Covariance Calculated
✅ Correlation Calulated


# Project Conclusion

This project applied **Inferential Statistics** techniques on a **Healthcare Dataset** to analyze relationships between different variables. Methoda such as **Confidence Intervals**, **Hypothesis Testing**, **t-Test**, **Chi-Square Test**, **ANOVA**, **Covariance**, and **Correlation** were used for statistical analysis.

The results helped identify **significant patterns**, **associations**, and **relationships** among health - related factors. Overall, the project demonstrated how **Statical methods** can be  used to make **data-driven decisions** and draw **meaningfull conclusions from real - world data